# Agent 第 12 课：AWS Bedrock Converse API + Tool Use

正式进入 AWS。这一课把 Phase 1 第 2 课（OpenAI function calling）那套 tool use，在 Bedrock 上重走一遍。

学完这节你能回答：
1. Bedrock 的 `converse()` 和 OpenAI 的 `chat.completions.create()` 有啥本质差异
2. Bedrock 上怎么声明工具、怎么拿到 tool_use 响应、怎么把结果回传
3. 跨模型（Claude / Nova / Llama）时 tool use 的行为差异在哪

> 需要 AWS 凭证 + Bedrock model access。环境变量 `AWS_REGION` 建议 `us-west-2`（模型最齐）。

## 1. 为什么 Bedrock 不直接兼容 OpenAI

AWS 的考量：
- Bedrock 背后有多个模型厂商（Anthropic / Meta / Mistral / Amazon Nova / Cohere …），每家原生格式都不一样
- 如果直接暴露每家的原生 API（`InvokeModel`），客户每换一家就得改代码
- 所以 AWS 出了一个**统一的 Converse API**：一套 message / tool / system 字段，后端自动翻译到各模型

代价：Converse 的字段名跟 OpenAI 只是"像"，不是"一样"。有几个关键差异下面一条一条讲。

## 2. Converse vs OpenAI —— 字段 diff 表

| 概念 | OpenAI | Bedrock Converse |
|---|---|---|
| 系统提示 | `messages[0] = {"role":"system",...}` | **独立字段** `system=[{"text": "..."}]` |
| 用户消息 | `{"role":"user","content":"..."}` | `{"role":"user","content":[{"text":"..."}]}` ← content 永远是 list |
| 工具声明 | `tools=[{"type":"function","function":{...}}]` | `toolConfig={"tools":[{"toolSpec":{...}}]}` |
| 模型要调工具 | 返回 `tool_calls[...]` | 返回 content block 里混着 `{"toolUse":{...}}` |
| 回传工具结果 | role="tool", `tool_call_id` | role="user", content 里 `{"toolResult":{"toolUseId": ..., "content":[...]}}` |
| 停止原因 | `finish_reason="tool_calls"` | `stopReason="tool_use"` |

一句话记：**Converse 的 content 永远是 block 数组**，每个 block 要么是 text、要么是 toolUse、要么是 toolResult。

In [ ]:
# 准备 boto3 客户端
import boto3, json, os
os.environ.setdefault("AWS_REGION", "us-west-2")

brt = boto3.client("bedrock-runtime")
MODEL = "anthropic.claude-sonnet-4-6-20260101-v1:0"  # 按账户可见 model id 调整
# 也可用 cross-region inference profile: "us.anthropic.claude-opus-4-7-20260101-v1:0"

resp = brt.converse(
    modelId=MODEL,
    system=[{"text": "You are a concise assistant. Reply in one sentence."}],
    messages=[{"role": "user", "content": [{"text": "Bedrock Converse 和 OpenAI chat 最大的区别是什么？"}]}],
    inferenceConfig={"maxTokens": 200, "temperature": 0.2},
)
print(resp["output"]["message"]["content"][0]["text"])
print("stopReason =", resp["stopReason"])
print("usage =", resp["usage"])

## 3. 声明工具：toolConfig

`toolConfig.tools[i].toolSpec` 里三个关键字段：
- `name`: 工具名
- `description`: 描述，**这里写得好坏直接决定模型调用得准不准**
- `inputSchema.json`: JSON schema，跟 OpenAI 一样

In [ ]:
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_weather",
                "description": "Get current weather for a city. Returns temperature in Celsius and a short condition string.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "city": {"type": "string", "description": "City name, e.g. Seattle"},
                        },
                        "required": ["city"],
                    }
                },
            }
        }
    ]
}

# 工具的"真"实现（mock）
def get_weather(city: str):
    data = {"Seattle": (11, "rainy"), "Beijing": (18, "smoggy"), "Tokyo": (15, "clear")}
    t, cond = data.get(city, (20, "unknown"))
    return {"city": city, "temp_c": t, "condition": cond}

print(json.dumps(tool_config, indent=2, ensure_ascii=False))

## 4. 一整个 tool use 回合

走完整流程：**user 发问 → 模型返回 toolUse → 我们执行 → 把 toolResult 回传 → 模型生成最终答复**。

注意 Converse 里 toolResult 是以 role="user" 发回去的（不是 "tool"）。

In [ ]:
def chat_with_tools(user_text):
    messages = [{"role": "user", "content": [{"text": user_text}]}]

    for turn in range(5):  # Phase 1 的 max_steps 还是要的
        resp = brt.converse(
            modelId=MODEL,
            system=[{"text": "You can call tools. Prefer calling a tool over guessing."}],
            messages=messages,
            toolConfig=tool_config,
            inferenceConfig={"maxTokens": 400, "temperature": 0},
        )

        out = resp["output"]["message"]
        messages.append(out)  # 把 assistant 回复也塞进 history

        stop = resp["stopReason"]
        print(f"  turn {turn}: stopReason={stop}")

        if stop != "tool_use":
            for blk in out["content"]:
                if "text" in blk:
                    return blk["text"]
            return "(no text)"

        # 找出所有 toolUse block，执行，组装 toolResult 回传
        tool_results = []
        for blk in out["content"]:
            if "toolUse" not in blk:
                continue
            tu = blk["toolUse"]
            print(f"    -> tool {tu['name']} args={tu['input']}")
            result = get_weather(**tu["input"]) if tu["name"] == "get_weather" else {"error": "unknown tool"}
            tool_results.append({
                "toolResult": {
                    "toolUseId": tu["toolUseId"],
                    "content": [{"json": result}],
                }
            })

        messages.append({"role": "user", "content": tool_results})
    return "(max turns reached)"

print("===")
print(chat_with_tools("今天北京天气怎么样？一句话告诉我。"))

## 5. 多工具 + 并行 tool use

Claude（和 Nova Pro）在 Converse 里会**一次返回多个 toolUse block**——并行调用。你的 dispatcher 必须能处理一个 assistant 消息里的多次 `toolUse`，上面的代码已经写成 for 循环所以天然支持。

小模型（Llama / 旧 Nova Lite）**不会并行**，一次只给一个 toolUse。写代码时不要假设并行一定发生，也不要假设一定不发生。

In [ ]:
# 加第二个工具：时间
tool_config_multi = {
    "tools": tool_config["tools"] + [
        {"toolSpec": {
            "name": "get_time",
            "description": "Get current local time for a city.",
            "inputSchema": {"json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }},
        }}
    ]
}

def get_time(city):
    tz = {"Seattle": "07:30", "Beijing": "22:30", "Tokyo": "23:30"}.get(city, "12:00")
    return {"city": city, "local_time": tz}

# 让模型一次要两个城市的天气 + 时间，观察它是否并行
messages = [{"role": "user", "content": [{"text": "Beijing 和 Tokyo 现在的天气和本地时间分别是什么？"}]}]
resp = brt.converse(modelId=MODEL, system=[{"text":"Use tools to answer."}],
                    messages=messages, toolConfig=tool_config_multi,
                    inferenceConfig={"maxTokens":500,"temperature":0})
blocks = resp["output"]["message"]["content"]
print("block 数量:", len(blocks))
for b in blocks:
    if "toolUse" in b:
        print("  toolUse:", b["toolUse"]["name"], b["toolUse"]["input"])
    elif "text" in b:
        print("  text:", b["text"][:80])

## 6. 跨模型行为差异（一个不踩就吃亏的点）

同样一段 toolConfig，换个模型可能表现差很大：

| 行为 | Claude Sonnet/Opus 4.x | Nova Pro | Llama 3.x | Mistral Large |
|---|---|---|---|---|
| 并行 tool use | 强 | 支持 | 不支持 | 部分 |
| 遵守 required 字段 | 稳 | 稳 | 常漏 | 常漏 |
| 遵守 enum / format | 稳 | 稳 | 偶尔 | 偶尔 |
| Tool description 超长（>500 tokens） | 能吃 | 能吃 | 容易忽略后半 | 容易忽略后半 |
| stopReason="end_turn" 时夹带工具描述的废话 | 几乎不 | 偶尔 | 常见 | 常见 |

**工程建议**：
- 选强模型（Claude / Nova Pro 起步）作为 agent 核心
- 选小模型（Haiku / Nova Lite）作为**子任务 worker**（单步、无 tool 或仅 1 个 tool）
- 在 eval set 里每次换模型都跑一遍（回到第 9 课）

## 7. 成本 / 速率的几个坑

- **`usage` 字段里是 token 数**，不是钱。自己乘价目表。
- Bedrock on-demand 默认有 **region-level TPM / RPM 限制**，Claude Opus 在 us-west-2 起步 200K TPM 左右，agent 很容易撞上——这时上 **Provisioned Throughput** 或切换 region。
- Cross-region inference profile（`us.anthropic...`）能自动 fail-over 到相邻 region，生产上强烈建议默认开。
- Tool use 的每一轮都是**完整对话重发**——history 每加一条都在重计费。Phase 1 的 "max_steps / max_tokens" 三把锁在 Bedrock 上一样适用。

---

## 8. 工程直觉

1. **Converse 的 content 永远是 block 列表**，记住这一条，90% 的字段不匹配错误能自己排。
2. **toolResult 以 role="user" 发**，不是 "tool"。
3. **跨 region inference profile** 是 agent 生产的默认配置，不是可选。
4. **toolSpec.description 是 agent 提示工程的主战场**——多迭代几版，用 eval 对比。
5. 下一课进**托管 Bedrock Agents**：不用自己写这个 while 循环，AWS 帮你跑，但代价是你失去一些细粒度控制。